# SpaceX Falcon 9 First-Stage Landing Prediction
## Applied Data Science Capstone — runnable starter notebook
This notebook loads an attributed public course dataset, inspects it, visualises landing outcomes, and compares four classification models. Run all cells in Google Colab or Jupyter. Your measured results may differ from the published summary in the presentation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, jaccard_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from IPython.display import display


## 1. Load the dataset
The CSV is hosted in a public capstone repository. Keep the attribution in your final project.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/IslamSaqer/Applied-Data-Science-Capstone/main/spacex_launch_dash.csv'
df = pd.read_csv(DATA_URL)
display(df.head())
print('Shape:', df.shape)
print(df.info())


## 2. Data quality and summary

In [ ]:
display(df.describe(include='all').T)
print('Missing values by column:')
display(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Landing class counts:')
display(df['class'].value_counts().sort_index())
print('Success rate (%):', round(df['class'].mean()*100, 1))


## 3. Exploratory visualisation

In [ ]:
ax = df['class'].value_counts().sort_index().plot(kind='bar', figsize=(6,4), title='Landing outcome counts')
ax.set_xlabel('Class (0 = non-success, 1 = success)')
ax.set_ylabel('Number of launches')
plt.tight_layout(); plt.show()

if 'Launch Site' in df.columns:
    site_rate = df.groupby('Launch Site')['class'].mean().sort_values(ascending=False)
    display((site_rate*100).round(1).rename('Success rate (%)').to_frame())
    site_rate.mul(100).plot(kind='bar', figsize=(8,4), title='Landing success rate by launch site', ylabel='Success rate (%)')
    plt.tight_layout(); plt.show()

if 'Payload Mass (kg)' in df.columns:
    print('Average payload mass (kg):', round(df['Payload Mass (kg)'].mean(), 1))
    df.boxplot(column='Payload Mass (kg)', by='class', figsize=(6,4))
    plt.title('Payload mass by landing class'); plt.suptitle(''); plt.xlabel('Class'); plt.tight_layout(); plt.show()


## 4. SQL-style aggregation with Pandas
These grouped summaries correspond to common SQL GROUP BY operations.

In [ ]:
print('Launches by site:')
display(df.groupby('Launch Site').agg(launches=('class','size'), success_rate=('class','mean')).assign(success_rate_pct=lambda x: (x.success_rate*100).round(1)).drop(columns='success_rate'))
print('Landing outcome counts:')
display(df.groupby('class').size().rename('launch_count').to_frame())


## 5. Prepare features and train/test split
Only pre-launch features are used here. The small course dataset is for learning, not production use.

In [ ]:
target = 'class'
candidate_features = ['Launch Site', 'Payload Mass (kg)', 'Booster Version Category']
features = [col for col in candidate_features if col in df.columns]
X = df[features].copy()
y = df[target].astype(int)
numeric = [col for col in features if pd.api.types.is_numeric_dtype(X[col])]
categorical = [col for col in features if col not in numeric]
numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocess = ColumnTransformer([('num', numeric_pipe, numeric), ('cat', categorical_pipe, categorical)])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print('Features:', features, '| Train:', X_train.shape, '| Test:', X_test.shape)


## 6. Compare four classifiers

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=2000),
    'SVM': SVC(),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=4),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}
results = []
for name, estimator in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', estimator)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    results.append({'Model': name, 'Accuracy': accuracy_score(y_test, pred), 'F1': f1_score(y_test, pred, zero_division=0), 'Jaccard': jaccard_score(y_test, pred, zero_division=0)})
    print('\n', name)
    print('Confusion matrix:\n', confusion_matrix(y_test, pred))
    print(classification_report(y_test, pred, zero_division=0))
results_df = pd.DataFrame(results).sort_values('F1', ascending=False)
display(results_df.round(3))


## 7. Interpretation and submission evidence
Add your own written interpretation below after running the notebook. Include screenshots of these outputs in your presentation. Do not copy the source-reported 100% SVM score as your own unless your run reproduces it.

- What features appear associated with landing success?
- Which model performs best on your held-out test set, and on which metric?
- What do the false positives and false negatives mean in this project?
- Why is a small dataset a limitation?
